# Failure Analysis — ImageTrust-AI

This notebook analyzes cases where the model makes incorrect predictions.
Understanding failures is as important as measuring accuracy.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from src.models.inference import load_model, predict
from src.data.loader import get_dataloaders
from src.data.transforms import val_transforms

In [ ]:
# Load model
model = load_model('../saved_models/best_model.pth')
model.eval()
print('Model loaded')

In [ ]:
# Load test set
_, _, test_loader = get_dataloaders(batch_size=32)
print(f'Test batches: {len(test_loader)}')

In [ ]:
# Collect wrong predictions
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = model.to(device)

wrong_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()

        wrong_mask = (preds != labels).squeeze()
        if wrong_mask.any():
            wrong_images = images[wrong_mask].cpu()
            wrong_labels = labels[wrong_mask].cpu()
            wrong_probs = probs[wrong_mask].cpu()
            for img, lbl, prob in zip(wrong_images, wrong_labels, wrong_probs):
                wrong_preds.append({
                    'image': img,
                    'true_label': int(lbl.item()),
                    'pred_prob': prob.item()
                })

print(f'Total wrong predictions: {len(wrong_preds)}')

In [ ]:
# Analyse confidence on wrong predictions
confidences = []
for wp in wrong_preds:
    prob = wp['pred_prob']
    confidence = prob if prob >= 0.5 else 1 - prob
    confidences.append(confidence)

confidences = np.array(confidences)
print(f'Mean confidence on wrong predictions: {confidences.mean():.4f}')
print(f'High confidence wrong (>90%): {(confidences > 0.9).sum()}')
print(f'Low confidence wrong (50-70%): {(confidences < 0.7).sum()}')

In [ ]:
# Plot confidence distribution of failures
plt.figure(figsize=(10, 4))
plt.hist(confidences, bins=20, color='tomato', edgecolor='black')
plt.xlabel('Confidence')
plt.ylabel('Count')
plt.title('Confidence Distribution on Wrong Predictions')
plt.axvline(x=0.9, color='black', linestyle='--', label='90% threshold')
plt.legend()
plt.tight_layout()
plt.savefig('failure_confidence_dist.png')
plt.show()
print('Saved plot')

In [ ]:
# Show sample failure cases
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor):
    return (tensor * std[:, None, None] + mean[:, None, None]).clamp(0, 1)

n_show = min(8, len(wrong_preds))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    wp = wrong_preds[i]
    img = denormalize(wp['image']).permute(1, 2, 0).numpy()
    true = 'Real' if wp['true_label'] == 0 else 'Fake'
    prob = wp['pred_prob']
    pred = 'Fake' if prob >= 0.5 else 'Real'
    conf = prob if prob >= 0.5 else 1 - prob
    ax.imshow(img)
    ax.set_title(f'True: {true}\nPred: {pred} ({conf:.0%})', fontsize=9)
    ax.axis('off')

plt.suptitle('Sample Failure Cases', fontsize=13)
plt.tight_layout()
plt.savefig('failure_samples.png')
plt.show()
print('Saved failure samples')

## Summary

Key findings from failure analysis:
- Note mean confidence on failures
- Note how many failures were high-confidence wrong predictions
- These are the most dangerous failures — model is wrong AND certain
- Real-world testing confirmed failures on unseen generators (DALL-E, MidJourney)

## Why failures happen
1. Model trained on limited generator types
2. Distribution shift from unseen generators
3. Heavily edited real images share visual patterns with AI images
4. Model learns generator-specific artifacts, not universal AI patterns